In [41]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder,LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense,Input, Embedding, Flatten, Dot, Concatenate


In [42]:
class Autoencoder(Model):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = Sequential([
            Dense(64, activation='relu'),
            Dense(32, activation='relu')
        ])
        self.decoder = Sequential([
            Dense(64, activation='relu'),
            Dense(input_dim, activation='sigmoid')
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [43]:
def build_ncf_model(num_users, num_items, embedding_size=50):
    user_input = Input(shape=(1,))
    item_input = Input(shape=(1,))

    user_embedding = Embedding(input_dim=num_users, output_dim=embedding_size)(user_input)
    item_embedding = Embedding(input_dim=num_items, output_dim=embedding_size)(item_input)

    user_vec = Flatten()(user_embedding)
    item_vec = Flatten()(item_embedding)

    concatenated = Concatenate()([user_vec, item_vec])
    x = Dense(64, activation='relu')(concatenated)
    x = Dense(32, activation='relu')(x)
    output = Dense(1, activation='linear')(x)

    model = Model(inputs=[user_input, item_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

In [44]:
product_data = pd.DataFrame({
    'item_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'name': ['Laptop', 'T-Shirt', 'Organic Apple', 'Smartphone', 'Headphones', 'Blender', 'Jeans', 'Milk'],
    'category': ['Electronics', 'Clothing', 'Groceries', 'Electronics', 'Electronics', 'Home Appliances', 'Clothing', 'Groceries'],
    'price': [800, 20, 3, 500, 100, 60, 40, 2],
    'brand': ['Brand A', 'Brand B', 'Brand C', 'Brand A', 'Brand D', 'Brand E', 'Brand B', 'Brand C'],
    'rating': [4.5, 3.2, 5.0, 2.7, 4.1, 3.9, 4.0, 4.8],
    'image_url': [
        "https://dummyimage.com/300x200/000/fff&text=Laptop",
        "https://dummyimage.com/300x200/000/fff&text=T-Shirt",
        "https://dummyimage.com/300x200/000/fff&text=Apple",
        "https://dummyimage.com/300x200/000/fff&text=Smartphone",
        "https://dummyimage.com/300x200/000/fff&text=Headphones",
        "https://dummyimage.com/300x200/000/fff&text=Blender",
        "https://dummyimage.com/300x200/000/fff&text=Jeans",
        "https://dummyimage.com/300x200/000/fff&text=Milk"
    ]
})


# User Dataset
user_data = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Ethan'],
    'age': [25, 30, 22, 28, 35],
    'location': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
})

# User-Item Interaction Dataset
user_item_data = pd.DataFrame({
    'user_id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
    'item_id': [101, 102, 101, 103, 102, 104, 105, 106, 107, 108],
    'rating': [5, 3, 4, 2, 5, 1, 4, 5, 3, 5]
})


In [45]:
def preprocess_content_data():
    content_data = product_data.copy()

    # Select columns to encode
    categorical_columns = ['category', 'brand']
    numeric_columns = ['price', 'rating']

    # One-Hot Encoding for categorical columns
    encoder = OneHotEncoder(sparse_output=False)
    encoded_data = encoder.fit_transform(content_data[categorical_columns])
    encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_columns))

    # Combine numerical and encoded columns
    content_features = pd.concat([content_data[numeric_columns], encoded_df], axis=1)
    content_features['item_id'] = content_data['item_id']

    return content_features

content_features = preprocess_content_data()


In [46]:
def content_based_recommendation(item_id: int, top_n: int = 2):
    # Get the item vector
    item_vector = content_features[content_features['item_id'] == item_id].drop('item_id', axis=1).values

    # # Ensure the item exists
    # if item_vector.size == 0:
    #     return error

    # Get all content vectors
    content_vectors = content_features.drop('item_id', axis=1).values

    # Compute cosine similarity
    similarities = cosine_similarity(item_vector, content_vectors)[0]

    # Get top N recommendations
    item_indices = np.argsort(similarities)[::-1][1:top_n + 1]
    recommendations = content_features.iloc[item_indices]['item_id'].tolist()

    # Fetch item details
    recommended_items = product_data[product_data['item_id'].isin(recommendations)][['item_id', 'name', 'image_url']]
    return recommended_items.to_dict(orient='records')


In [47]:

# Collaborative Filtering with Autoencoder
# Creates a matrix with users as rows, items as columns, and ratings as values.
user_item_matrix = user_item_data.pivot(index='user_id', columns='item_id', values='rating').fillna(0)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(user_item_matrix)
print(scaled_data.shape[1])

# Gets the number of items (columns) — i.e., how many items each user is rating.
input_dim = scaled_data.shape[1]
autoencoder = Autoencoder(input_dim=input_dim)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(scaled_data, scaled_data, epochs=50, batch_size=2, verbose=0)


8


In [48]:
def collaborative_recommendation(user_id: int, top_n: int = 2):
    try:
        user_index = user_item_matrix.index.get_loc(user_id)
        user_data = scaled_data[user_index].reshape(1, -1)
        predictions = autoencoder(user_data).numpy()[0]
        top_indices = np.argsort(predictions)[::-1][:top_n]
        recommended_items = user_item_matrix.columns[top_indices].tolist()
        recommended_items_data = product_data[product_data['item_id'].isin(recommended_items)][['item_id', 'name', 'image_url']]
        return recommended_items_data.to_dict(orient='records')
    except KeyError:
        return []

In [49]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

user_item_data['user'] = user_encoder.fit_transform(user_item_data['user_id'])
user_item_data['item'] = item_encoder.fit_transform(user_item_data['item_id'])

num_users = user_item_data['user'].nunique()
num_items = user_item_data['item'].nunique()

ncf_model = build_ncf_model(num_users, num_items)

X = [user_item_data['user'].values, user_item_data['item'].values]
y = user_item_data['rating'].values

ncf_model.fit(X, y, epochs=20, batch_size=4, verbose=0)

In [50]:
def ncf_recommendation(user_id: int, top_n: int = 2):
    if user_id not in user_data['user_id'].values:
        return []

    user_idx = user_encoder.transform([user_id])[0]

    # Get all item indices
    all_item_ids = product_data['item_id'].values
    all_item_indices = item_encoder.transform(all_item_ids)

    # Prepare input
    user_input = np.full_like(all_item_indices, user_idx)
    predictions = ncf_model.predict([user_input, all_item_indices], verbose=0).flatten()

    # Exclude already rated items
    rated_items = user_item_data[user_item_data['user_id'] == user_id]['item_id'].values
    unrated_mask = ~np.isin(all_item_ids, rated_items)

    recommended_indices = np.argsort(predictions[unrated_mask])[::-1][:top_n]
    recommended_items = all_item_ids[unrated_mask][recommended_indices]

    return product_data[product_data['item_id'].isin(recommended_items)][['item_id', 'name', 'image_url']].to_dict(orient='records')



In [51]:
def get_item(item_id: int):

    item = product_data[product_data['item_id'] == item_id].to_dict(orient='records')

    return item[0]

In [52]:
def get_user(user_id: int):
    user = user_data[user_data['user_id'] == user_id].to_dict(orient='records')

    return user[0]

In [61]:
def recommend_content(request):
    return {"recommendations": content_based_recommendation(request['item_id'])}

def recommend_collaborative(request):
    return {"recommendations": collaborative_recommendation(request['user_id'])}

def recommend_mixed(request):

    content_recs = content_based_recommendation(request['item_id'])
    collab_recs = collaborative_recommendation(request['user_id'])
    combined_recs = {item['item_id']: item for item in content_recs + collab_recs}
    return {"recommendations": list(combined_recs.values())}

def recommend_ncf(request):
    return {"recommendations": ncf_recommendation(request['user_id'])}


In [62]:
request = {
    "user_id": 1,
    "item_id": 101
}


In [63]:
recommend_content(request)

{'recommendations': [{'item_id': 104,
   'name': 'Smartphone',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=Smartphone'},
  {'item_id': 105,
   'name': 'Headphones',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=Headphones'}]}

In [64]:
recommend_collaborative(request)

{'recommendations': [{'item_id': 101,
   'name': 'Laptop',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=Laptop'},
  {'item_id': 102,
   'name': 'T-Shirt',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=T-Shirt'}]}

In [65]:
recommend_ncf(request)

{'recommendations': [{'item_id': 106,
   'name': 'Blender',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=Blender'},
  {'item_id': 108,
   'name': 'Milk',
   'image_url': 'https://dummyimage.com/300x200/000/fff&text=Milk'}]}